# สถาปัตยกรรมระบบการกู้คืนข้อมูลและการสร้างคำตอบในบริบทเฉพาะด้าน
### (Architectural Framework for Domain-Specific Retrieval-Augmented Generation: RAG)
โครงการนี้มุ่งเน้นการสร้างระบบ RAG ที่มีประสิทธิภาพสูงเพื่อรองรับการประมวลผลคำตอบจากฐานข้อมูลสินค้าและนโยบายร้านค้า โดยมีสถาปัตยกรรมที่โดดเด่นแบ่งออกเป็น 5 แกนหลักดังนี้:
1.  **Structural Data Partitioning & Multi-level Chunking:** การระบุโครงสร้างเอกสารและการจัดแบ่งส่วนข้อมูลเชิงลำดับชั้น (Hierarchical Recursive Chunking) โดยมีการกำหนดขนาดของข้อมูล (Chunk Size) และการซ้อนทับ (Overlap) ที่เหมาะสม เพื่อรักษาความต่อเนื่องของเนื้อหาและโครงสร้างเชิงลอจิกของเอกสารต้นฉบับ
2.  **Contextual Retrieval Augmentation:** การประยุกต์ใช้เทคนิค **Contextual Retrieval** เพื่อเสริมสร้างความสัมพันธ์เชิงบริบท (Semantic Enrichment) โดยการใช้โมเดลภาษาขนาดใหญ่ (LLM) สรุปสาระสำคัญของเอกสารแม่และฝังลงในข้อมูลส่วนย่อยทุกชิ้น เพื่อแก้ปัญหาการสูญเสียบริบทเมื่อมีการหั่นย่อยข้อมูล (Context Fragmentation)
3.  **Unified Hybrid Store Architecture:** การสร้างระบบจัดเก็บและสืบค้นข้อมูลแบบผสมผสานบนพื้นฐานของ **Unified Contextual Chunks** โดยมีการจัดทำดัชนีระบุตำแหน่งข้อมูลทั้งในมิติคำศัพท์ (Lexical - BM25) และมิติความหมายเชิงลึก (Neural/Dense - FAISS) เพื่อให้ผลลัพธ์การสืบค้นมีความสอดคล้องกันระหว่างระบบทั้งสอง (Retrieval Consistency)
4.  **Adaptive Query Refinement & Keyword Extraction:** การใช้โมเดลภาษาในการวิเคราะห์และปรับแต่งคำถาม (Query Rewrite) รวมถึงการสกัดเอาสาระสำคัญและคำค้นหาเชิงลึก (Feature Extraction) เพื่อเพิ่มความแม่นยำในการเข้าถึงข้อมูลจากระบบดัชนีแบบผสมผสาน
5.  **Chain-of-Thought Response Orchestration:** การควบคุมกระบวนการประมวลผลคำตอบ (Response Orchestration) ผ่านเทคนิคการเรียนรู้ผ่านตัวอย่างเชิงบริบท (Few-shot Prompting) และการบังคับโครงสร้างลำดับความคิด (Chain-of-Thought) เพื่อเพิ่มขีดความสามารถในการตัดสินใจและการคำนวณที่ซับซ้อนให้แก่โมเดลภาษาขนาดใหญ่
---


## 💡 เจาะลึกเทคนิคพิเศษ: Contextual Retrieval Strategy
**ปัญหาหลักของการทำ RAG ทั่วไป (The Semantic Gap):**
เมื่อเราทำการหั่นข้อความเป็นส่วนย่อย (Chunking) เพื่อให้โมเดลประมวลผลได้ บริบทสำคัญของเอกสารมักจะสูญหายไป เช่น ข้อความส่วนย่อยอาจระบุเพียงราคาและสเปก แต่ไม่ได้ระบุว่าเป็นของสินค้าชิ้นใด เนื่องจากชื่อสินค้าอยู่ข้างบนยอดของเอกสาร
**โซลูชันที่นำมาใช้ (The Contextual Solution):**
เรานำเทคนิค **Contextual Retrieval (จาก Anthropic)** มาประยุกต์ใช้ โดยการจัดเตรียมข้อมูลล่วงหน้า (Pre-computational Stage) ดังนี้:
1. ให้โมเดลภาษา (LLM) อ่านเอกสารทั้งฉบับเพื่อทำความเข้าใจภาพรวม
2. สรุปบริบทที่สำคัญของเอกสารนั้น (เช่น ชื่อรุ่น, หมวดหมู่, จุดเด่นหลัก)
3. นำสรุปดังกล่าวไปแปะไว้ที่ส่วนต้นของทุก Chunk ข้อมูลที่ถูกหั่นออกมา
**ผลลัพธ์เชิงสถาปัตยกรรม:**
วิธีนี้ช่วยให้ระบบค้นหา (Retrieval Engine) ทั้งแบบ Keyword และ Vector สามารถระบุตำแหน่งของข้อมูลที่ต้องการได้อย่างถูกต้องแม่นยำสูงสุด แม้ข้อมูลนั้นจะมีความคล้ายคลึงกันทางสเปกเพียงใดก็ตาม
---


## 📦 1. การจัดเตรียมสภาพแวดล้อมและโครงสร้างพื้นฐาน (Infrastructure Setup)
การติดตั้งไลบรารีมาตรฐานสำหรับการประมวลผลภาษาธรรมชาติและการจัดการดัชนีระบุตำแหน่งข้อมูล (Indexing) เพื่อรองรับการทำงานของระบบอย่างครบวงจร


In [56]:
%pip install transformers torch accelerate sentence-transformers faiss-cpu pandas langchain langchain-text-splitters rank_bm25 pythainlp


Note: you may need to restart the kernel to use updated packages.


## 📚 2. การนำเข้าส่วนประกอบและโมเดลประมวลผล (System Library Ingestion)
การเตรียมเครื่องมือสำหรับการวิเคราะห์ข้อมูลและเทคโนโลยีการสืบค้นแบบเวกเตอร์ (Vector Space Modeling) รวมถึงระบบตัดคำภาษาไทยที่เหมาะสมกับบริบท


In [57]:
import os
import re
import glob
import pandas as pd
import numpy as np
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi
import warnings
warnings.filterwarnings('ignore')
print('✅ Import เสร็จเรียบร้อย!')


✅ Import เสร็จเรียบร้อย!


## 🤖 3. การกำหนดค่าพารามิเตอร์ของโมเดลภาษา (LLM Configuration)
การเลือกใช้โมเดลภาษาขนาดใหญ่ (LLM) ที่รองรับการประมวลผลคำตอบเชิงเหตุผล (Logical Reasoning) และการกำหนดจุดเชื่อมต่อ (End-point) สำหรับการประมวลผล


In [58]:
import requests
THAILLM_API_KEY = "9AN3RpP84wfbyVJoy1Fz8dIG3egjpm66"
THAILLM_MODEL = "openthaigpt"  # หรือ openthaigpt, kbtg, pathumma
print(f'✅ ตั้งค่า ThaiLLM API เรียบร้อย! (model: {THAILLM_MODEL})')


✅ ตั้งค่า ThaiLLM API เรียบร้อย! (model: openthaigpt)


## 📶 4. โปรโตคอลการสื่อสารและการจัดการข้อผิดพลาด (Robust Communication Protocol)
การสร้างฟังก์ชันอินเทอร์เฟซสำหรับการรับ-ส่งข้อมูลกับโมเดล โดยมีกลไกควบคุมความเสถียร (Error Handling) และการจัดการปริมาณการเรียกใช้ (Rate Limiting)


In [59]:
import json
# ฟังก์ชันหัวใจหลักในการคุยกับ API: รองรับระบบการตอบแบบใช้ความคิด (Think)
# ฟังก์ชันอินเทอร์เฟซสำหรับการวิเคราะห์หาคำตอบ (Response Interface) รองรับระบบ Chain-of-Thought
def ask_llm(messages, max_new_tokens=512, temperature=0.0):
    """ส่ง messages ไปถาม ThaiLLM ผ่าน API"""
    url = f"http://thaillm.or.th/api/{THAILLM_MODEL}/v1/chat/completions"
    headers = {
        "Content-Type": "application/json; charset=utf-8",
        "apikey": THAILLM_API_KEY
    }
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": max_new_tokens,
        "temperature": temperature,
    }
    for attempt in range(5):
        try:
            # ใช้ data แทน json เพื่อบังคับ UTF-8
            resp = requests.post(
                url,
                headers=headers,
                data=json.dumps(payload, ensure_ascii=False).encode('utf-8'),
                timeout=120
            )
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"  ⏳ Rate limited, waiting {wait}s...")
                import time; time.sleep(wait)
                continue
            # เช็คสถานะการตอบจาก Server: ถ้า error จะเข้าสู่วงจร Retry
            # การตรวจสอบสถานะการตอบสนองและระบบควบคุมความผิดพลาด (Error Handling)
            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except Exception as e:
            import time
            wait = 2 ** attempt
            print(f"  ❌ Error: {e}, retrying in {wait}s...")
            time.sleep(wait)
    return ""
# ทดสอบ
response = ask_llm([{"role": "user", "content": "สวัสดีครับ คุณคือใคร?"}])
print('✅ ทดสอบ LLM:', response[:300])


✅ ทดสอบ LLM: <think>
คำถามนี้เป็นคำถามเกี่ยวกับตัวตนของระบบ AI ดังนั้นต้องอธิบายที่มาและวัตถุประสงค์

กระบวนการตอบ:
1. แนะนำตัว
2. อธิบายที่มา
3. แจ้งวัตถุประสงค์
</think>

สวัสดีครับ ผมคือ **OpenThaiGPT** ผู้ช่วยอัจฉริยะที่พัฒนาขึ้นมาเพื่อคนไทยโดยเฉพาะ

**ข้อมูลเบื้องต้น:**
- **ชื่อ:** OpenThaiGPT
- **นามแฝง:**


## 📂 ขั้นตอนที่ 5: การประมวลผลเอกสารและการแบ่งส่วนข้อมูล (Structural Data Partitioning)
**[Architecture Point 1: Multi-level Chunking]**  
เริ่มต้นกระบวนการด้วยการนำเข้าเอกสารจากฐานข้อมูล และใช้หลักการ **Recursive Structural Chunking** เพื่อจัดแบ่งย่อยข้อมูลโดยยังคงรักษาความสัมพันธ์เชิงลอจิกของเนื้อหาผ่านการกำหนด Overlap ที่เหมาะสม


In [60]:
KB_DIR = 'data/knowledge_base'
# อ่านไฟล์ทั้งหมด
documents = []  # list of {"text": ..., "source": ..., "category": ...}
for category in ['products', 'policies', 'store_info']:
    pattern = os.path.join(KB_DIR, category, '*.md')
    for filepath in sorted(glob.glob(pattern)):
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        filename = os.path.basename(filepath)
        documents.append({
            'text': content,
            'source': filename,
            'category': category
        })
print(f'📂 อ่านไฟล์ทั้งหมด: {len(documents)} ไฟล์')
print(f'   - products: {sum(1 for d in documents if d["category"]=="products")}')
print(f'   - policies: {sum(1 for d in documents if d["category"]=="policies")}')
print(f'   - store_info: {sum(1 for d in documents if d["category"]=="store_info")}')
# Chunking ด้วย RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n## ', '\n### ', '\n#### ', '\n\n', '\n', ' ', '']
)
all_chunks = []   # text ของแต่ละ chunk
all_sources = []  # ชื่อไฟล์ต้นทาง
all_metadata = [] # metadata เพิ่มเติม
for doc in documents:
    # เอาชื่อสินค้า/นโยบายใส่ไว้หัว chunk เพื่อให้ retrieval แม่นขึ้น
    header = f"[แหล่งข้อมูล: {doc['source']} | หมวด: {doc['category']}]\n"
    chunks = text_splitter.split_text(doc['text'])
    for chunk in chunks:
        all_chunks.append(header + chunk)
        all_sources.append(doc['source'])
        all_metadata.append({'source': doc['source'], 'category': doc['category']})
print(f'✅ Chunking เสร็จ: {len(all_chunks)} chunks จาก {len(documents)} ไฟล์')


📂 อ่านไฟล์ทั้งหมด: 118 ไฟล์
   - products: 110
   - policies: 5
   - store_info: 3
✅ Chunking เสร็จ: 795 chunks จาก 118 ไฟล์


## 💡 ขั้นตอนที่ 5.5: เทคนิคการเพิ่มความสัมพันธ์เชิงบริบท (Contextual Augmentation Strategy)
**[Architecture Point 2: Contextual Retrieval Augmentation]**  
ขั้นตอนนี้คือหัวใจสำคัญของการแก้ปัญหา Context Fragmentation โดยการใช้ LLM ในการสรุปบริบทองค์รวมของเอกสารต้นฉบับ แล้วนำมาฉีด (Inject) ลงไปในทุกส่วนย่อยของข้อมูล เพื่อเพิ่มความแม่นยำในขั้นตอนการสืบค้น


In [61]:
# === True Contextual Retrieval ===
# ใช้ LLM (Pathumma/OpenThaiGPT) อ่านเอกสารแม่คู่กับแต่ละ Chunk แล้วสรุปบริบทสั้นๆ แปะกลับเข้าไป
# เพื่อให้ระบบ Search ดึงข้อมูลแม่นยำขึ้น (Anthropic Contextual Retrieval Technique)
import time as _time
CACHE_FILE = 'data/contextual_chunks.json'
def generate_chunk_context(doc_text, chunk_text, filename):
    """ให้ LLM สรุปบริบทสั้นๆ ว่า Chunk นี้พูดถึงอะไร"""
    prompt = f"""<instruction>
คุณเป็นหุ่นยนต์ช่วยสรุปบริบท (Context)
ดูที่เอกสารต้นฉบับทั้งหมดด้านล่าง และดูที่เนื้อหาย่อย (Chunk) ที่ตัดมา
จงสรุปสั้นๆ (ไม่เกิน 2 ประโยค) ว่าเนื้อหาย่อยนี้กำลังพูดถึงสินค้าชื่ออะไร รุ่นอะไร และมีสรรพคุณ/เงื่อนไขอะไรซ่อนอยู่
ห้ามทวนคำสั่ง ตอบมาเป็นบริบทสั้นๆ เพียวๆ เท่านั้น
</instruction>
<source>{filename}</source>
<document>{doc_text[:2000]}</document>
<chunk>{chunk_text}</chunk>
"""
    messages = [{'role': 'user', 'content': prompt}]
    res = ask_llm(messages, max_new_tokens=200, temperature=0.0)
    clean = re.sub(r'<think>.*?</think>', '', res, flags=re.DOTALL).strip()
    return clean
# สร้าง mapping จาก source -> doc text
doc_map = {d['source']: d['text'] for d in documents}
total_needed = len(all_chunks)
# เช็คว่ามี Cache ครบหรือยัง (ถ้าครบแล้วข้าม / ถ้าไม่ครบทำต่อจากจุดที่ค้าง)
ctx_chunks = []
ctx_sources = []
ctx_metadatas = []
start_idx = 0
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, 'r', encoding='utf-8') as f:
        ctx_data = json.load(f)
    if len(ctx_data['chunks']) >= total_needed:
        # ครบแล้ว! โหลดจาก Cache เลย
        all_chunks = ctx_data['chunks']
        all_sources = ctx_data['sources']
        all_metadata = ctx_data['metadatas']
        print(f'🌟 [Contextual Mode] โหลดจาก Cache สำเร็จ: {len(all_chunks)} chunks (ครบแล้ว!)')
        start_idx = -1  # flag ว่าไม่ต้องทำอะไรแล้ว
    else:
        # ยังไม่ครบ! ทำต่อจากจุดที่ค้าง
        ctx_chunks = ctx_data['chunks']
        ctx_sources = ctx_data['sources']
        ctx_metadatas = ctx_data['metadatas']
        start_idx = len(ctx_chunks)
        print(f'🔄 พบ Cache เดิม {start_idx}/{total_needed} chunks — ทำต่อจากจุดที่ค้าง!')
if start_idx >= 0:
    if start_idx == 0:
        print(f'🚀 เริ่ม Contextual Retrieval: {total_needed} chunks (ใช้เวลา ~30-60 นาที)')
    
    for i in range(start_idx, total_needed):
        chunk = all_chunks[i]
        source = all_sources[i]
        meta = all_metadata[i]
        
        print(f'  [{i+1}/{total_needed}] {source}...', end='', flush=True)
        doc_text = doc_map.get(source, '')
        context = generate_chunk_context(doc_text, chunk, source)
        enriched = f'[{context}]\n{chunk}'
        ctx_chunks.append(enriched)
        ctx_sources.append(source)
        ctx_metadatas.append(meta)
        print(' ✅')
        
        # Checkpoint ทุก 20 chunks
        if (i + 1) % 20 == 0:
            with open(CACHE_FILE, 'w', encoding='utf-8') as f:
                json.dump({'chunks': ctx_chunks, 'sources': ctx_sources, 'metadatas': ctx_metadatas}, f, ensure_ascii=False)
            print(f'  💾 Checkpoint saved ({i+1}/{total_needed})')
    
    # Save final
    with open(CACHE_FILE, 'w', encoding='utf-8') as f:
        json.dump({'chunks': ctx_chunks, 'sources': ctx_sources, 'metadatas': ctx_metadatas}, f, ensure_ascii=False)
    
    all_chunks = ctx_chunks
    all_sources = ctx_sources
    all_metadata = ctx_metadatas
    print(f'\n✅✅ Contextual Retrieval เสร็จสมบูรณ์! {len(all_chunks)} chunks พร้อมบริบท')


🔄 พบ Cache เดิม 20/795 chunks — ทำต่อจากจุดที่ค้าง!
  [21/795] DN-DT-002_daonuea_tower_x10_max.md...  ❌ Error: 502 Server Error: Bad Gateway for url: http://thaillm.or.th/api/openthaigpt/v1/chat/completions, retrying in 1s...
 ✅
  [22/795] DN-DT-002_daonuea_tower_x10_max.md... ✅
  [23/795] DN-DT-002_daonuea_tower_x10_max.md... ✅
  [24/795] DN-DT-002_daonuea_tower_x10_max.md... ✅
  [25/795] DN-DT-002_daonuea_tower_x10_max.md... ✅
  [26/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [27/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [28/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [29/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [30/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [31/795] DN-DT-003_daonuea_mini_pc_m1.md... ✅
  [32/795] DN-DT-004_daonuea_all_in_one_27.md... ✅
  [33/795] DN-DT-004_daonuea_all_in_one_27.md... ✅
  [34/795] DN-DT-004_daonuea_all_in_one_27.md... ✅
  [35/795] DN-DT-004_daonuea_all_in_one_27.md... ✅
  [36/795] DN-DT-004_daonuea_all_in_one_27.md... ✅
  [37/795] DN-DT-004_dao

## 📦 ขั้นตอนที่ 6: การสร้างดัชนีระบุตำแหน่งข้อมูลแบบบูรณาการ (Hybrid Search Indexing)
**[Architecture Point 3: Unified Hybrid Store]**  
การสร้างระบบจัดเก็บข้อมูลแบบคู่ขนาน (Dual-Indexing) บนพื้นฐานของ **Unified Contextual Chunks** โดยประกอบด้วย FAISS (Dense Index) สำหรับความสัมพันธ์เชิงความหมาย และ BM25 (Lexical Index) สำหรับความสัมพันธ์เชิงคำศัพท์


In [62]:
import numpy as np
import faiss
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
print("⏳ กำลังเตรียมสมองส่วนค้นหา...")
embed_model = SentenceTransformer('intfloat/multilingual-e5-large-instruct')
# 1. FAISS
passage_texts = ['passage: ' + t for t in all_chunks]
print("⏳ กำลังสร้าง Dense Vectors...รอแป๊บ")
embeddings = embed_model.encode(passage_texts, show_progress_bar=True, batch_size=16)
embeddings = np.array(embeddings).astype('float32')
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
faiss_index.add(embeddings)
print("✅ สร้าง FAISS เสร็จแล้ว!")
# 2. BM25
print("⏳ กำลังสร้าง BM25...")
tokenized_corpus = [word_tokenize(doc, engine="newmm") for doc in all_chunks]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ สร้าง BM25 เสร็จแล้ว!")


⏳ กำลังเตรียมสมองส่วนค้นหา...
⏳ กำลังสร้าง Dense Vectors...รอแป๊บ


Batches:   0%|          | 0/50 [00:00<?, ?it/s]

✅ สร้าง FAISS เสร็จแล้ว!
⏳ กำลังสร้าง BM25...
✅ สร้าง BM25 เสร็จแล้ว!


## 🔍 ขั้นตอนที่ 7: กลไกการเรียกคืนข้อมูลและการจัดลำดับ (Retrieval Orchestration)
**[Architecture Point 3: Unified Hybrid Store]**  
การผสานผลลัพธ์จากดัชนีทั้งสองรูปแบบ (Hybrid Retrieval) พร้อมกระบวนการปรับมาตรฐานคะแนน (Score Normalization) เพื่อระบุชุดข้อมูลที่เกี่ยวข้องที่สุดสำหรับนำไปประมวลผลคำตอบ


In [63]:
def retrieve_dense(query, top_k=15):
    """Dense retrieval ด้วย FAISS (vector similarity)"""
    query_vec = embed_model.encode(['query: ' + query], normalize_embeddings=True)
    query_vec = np.array(query_vec).astype('float32')
    scores, indices = faiss_index.search(query_vec, top_k)
    return [(int(indices[0][i]), float(scores[0][i])) for i in range(len(indices[0]))]
def retrieve_sparse(query, top_k=15):
    """Sparse retrieval ด้วย BM25 (keyword matching)"""
    tokenized_query = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in top_indices]
# ระบบรวมผลการค้นหา: ดึงข้อดีของทั้ง Keyword Match และ Semantic Logic
# สถาปัตยกรรมระบบสืบค้นข้อมูลแบบผสมผสาน (Hybrid Retrieval Architecture)
def hybrid_retrieve(query, top_k=15, dense_weight=0.5, sparse_weight=0.5):
    """
    Hybrid retrieval: รวม Dense + Sparse scores
    dense_weight + sparse_weight = 1.0
    """
    # Dense
    # 1. ค้นหาด้วยความหมาย (กบ  -> สระน้ำ)
    # 1. การเรียกคืนข้อมูลเชิงความหมาย (Semantic Similarity Search)
    dense_results = retrieve_dense(query, top_k=top_k * 2)
    # Sparse
    # 2. ค้นหาด้วยคำสำคัญ (กบ -> "กบ")
    # 2. การเรียกคืนข้อมูลเชิงคำสำคัญ (Lexical Match Search)
    sparse_results = retrieve_sparse(query, top_k=top_k * 2)
    # Normalize scores to [0, 1]
    def normalize(results):
        if not results:
            return {}
        scores = [s for _, s in results]
        min_s, max_s = min(scores), max(scores)
        rng = max_s - min_s if max_s != min_s else 1.0
        return {idx: (s - min_s) / rng for idx, s in results}
    dense_norm = normalize(dense_results)
    sparse_norm = normalize(sparse_results)
    # Combine scores
    all_indices = set(dense_norm.keys()) | set(sparse_norm.keys())
    combined = []
    for idx in all_indices:
        score = (dense_weight * dense_norm.get(idx, 0.0) +
                 sparse_weight * sparse_norm.get(idx, 0.0))
        combined.append((idx, score))
    # Sort by combined score descending
    combined.sort(key=lambda x: x[1], reverse=True)
    return combined[:top_k]
# ทดสอบ
test_results = hybrid_retrieve('Watch S3 Ultra กันน้ำ', top_k=5)
print('✅ ทดสอบ Hybrid Retrieval:')
for idx, score in test_results:
    print(f'   score={score:.3f} | {all_sources[idx][:50]}...')


✅ ทดสอบ Hybrid Retrieval:
   score=1.000 | WK-SW-001_wongkhojon_watch_s3_ultra.md...
   score=0.483 | WK-SW-001_wongkhojon_watch_s3_ultra.md...
   score=0.473 | WK-SW-001_wongkhojon_watch_s3_ultra.md...
   score=0.457 | WK-SW-001_wongkhojon_watch_s3_ultra.md...
   score=0.340 | WK-SW-001_wongkhojon_watch_s3_ultra.md...


## ✍️ ขั้นตอนที่ 8: กระบวนการเกลาคำถามและสกัดคำสำคัญ (Adaptive Query Refinement)
**[Architecture Point 4: Query Extraction & Refinement]**  
การเพิ่มคุณภาพของคำถามต้นฉบับด้วยโมเดลภาษา เพื่อกำจัด Noise และระบุคีย์เวิร์ดสำคัญ (Keyword Extraction) สำหรับการทำ Sparse Search ให้มีความแม่นยำสูงสุด


In [64]:
def rewrite_query(question):
    """
    ใช้ ThaiLLM rewrite คำถามให้กระชับ เอาแต่ใจความสำคัญ
    เพื่อช่วยให้ retrieval แม่นขึ้น
    """
    prompt = """<instruction>
คุณเป็นผู้ช่วยสรุปคำถาม ให้สรุปคำถามต่อไปนี้ให้กระชับ เอาเฉพาะใจความสำคัญที่เกี่ยวกับสินค้าหรือบริการ
ตัดส่วนที่เป็นเรื่องเล่าหรือบริบทที่ไม่จำเป็นออก
ตอบเป็นคำถามสั้นๆ 1 ประโยค
</instruction>
<question>
""" + question + """
</question>
<rewritten_query>"""
    messages = [{"role": "user", "content": prompt}]
    rewritten = ask_llm(messages, max_new_tokens=100, temperature=0.0)
    # ทำความสะอาด
    rewritten = rewritten.split('</rewritten_query>')[0].strip()
    rewritten = rewritten.split('\n')[0].strip()
    return rewritten if len(rewritten) > 5 else question
# คำถามที่ไม่เกี่ยวข้องกับร้านเลย (hardcode เพื่อความแม่น)
OUT_OF_SCOPE_KEYWORDS = [
    'วันหยุดราชการ', 'ตั๋วเครื่องบิน', 'ดอกเบี้ยเงินฝาก',
    'สูตรผัดกระเพรา', 'สูตรอาหาร', 'โรงแรม', 'ร้านอาหาร'
]
def is_out_of_scope(question):
    """เช็คว่าคำถามไม่เกี่ยวข้องกับฐานข้อมูลของร้านเลย → ตอบ 10"""
    for kw in OUT_OF_SCOPE_KEYWORDS:
        if kw in question:
            return True
    return False
print('✅ Query Rewriting + Out-of-Scope Detection พร้อมใช้งาน!')
# ทดสอบ
test_q = 'เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่ยวภูเก็ต อยากรู้ว่าสายฟ้า Rugged R1 กันน้ำระดับไหนคะ'
print(f'Original: {test_q[:60]}...')
print(f'Rewritten: {rewrite_query(test_q)}')


✅ Query Rewriting + Out-of-Scope Detection พร้อมใช้งาน!
Original: เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่ยวภูเ...
Rewritten: <think>


## 🎯 9. วิศวกรรมคำสั่งและการประมวลผลคำตอบเชิงตรรกะ (Prompt & Logic Engineering & CoT)
การออกแบบคำสั่ง (Prompt Engineering) ที่ใช้กลไกการเรียนรู้ผ่านตัวอย่าง (Few-shot) และกระบวนการคิดวิเคราะห์ขั้นตอน (Chain-of-Thought) เพื่อการตัดสินใจเลือกคำตอบที่ถูกต้องที่สุด


In [ ]:
# ฟังชันก์ตัดสินใจตอบคำถาม: วาง Context, ใส่ Few-shot, และสั่งให้ LLM คิด
# กระบวนการวิเคราะห์และสังเคราะห์คำตอบ (Logic & Response Processing)
def answer_mcq(question_row, top_k=15, use_rewrite=False):
    """ตอบคำถามแบบ Multiple Choice โดยใช้ RAG (โหมดวิเคราะห์ลึก)"""
    qid = question_row['id']
    question = question_row['question']
    choices = []
    for i in range(1, 11):
        col = f'choice_{i}'
        if pd.notna(question_row.get(col)):
            choices.append(str(question_row[col]))
    search_query = question
    # Simple Keyword Extraction for Sparse Search (Remove common noise words)
    noise_words = ['เฮ้ยย', 'อ่านเยอะมากเลย', 'ขอถามนิดนึงนะคะ', 'คือพอดีว่าจะไป', 'ช่วยดูให้หน่อยได้ไหมครับ', 'เรื่องมันยาวนิดนึง']
    sparse_query = question
    for nw in noise_words:
        sparse_query = sparse_query.replace(nw, '')
    
    dense_res = retrieve_dense(search_query, top_k=top_k*2)
    sparse_res = retrieve_sparse(sparse_query, top_k=top_k*2)
    
    def norm(res):
        if not res: return {}
        sc = [s for _, s in res]
        mn, mx = min(sc), max(sc)
        rng = mx - mn if mx != mn else 1.0
        return {idx: (s - mn)/rng for idx, s in res}
        
    dn, sn = norm(dense_res), norm(sparse_res)
    comb = []
    for idx in set(dn.keys()) | set(sn.keys()):
        comb.append((idx, 0.5*dn.get(idx, 0.0) + 0.5*sn.get(idx, 0.0)))
    comb.sort(key=lambda x: x[1], reverse=True)
    results = comb[:top_k]
    # 1. เตรียมเนื้อหาอ้างอิงและจัดการ Token Limit
        # 1. การเตรียมคลังข้อมูลอ้างอิงและจำกัดจำนวนโทเคน (Contextual Framing)
        context_parts = []
    seen = set()
    for idx, float_score in results[:top_k]:
        src = all_sources[idx]
        context_parts.append(f"[{src} | score={float_score:.4f}]{all_chunks[idx]}")
        seen.add(src)
    context = '\n\n---\n\n'.join(context_parts)
    # ตัดบริบทไม่ให้เกิน Token Limit ของ API
    if len(context) > 6000:
        context = context[:6000]
    choices_text = '\n'.join([f'  {i+1}. {c}' for i, c in enumerate(choices)])
    # 2. ป้อน Prompt ที่ออกแบบมาเพื่อดักแก้โจทย์เลขและคำถามกวนประสาท
    # 2. การกำหนดวิศวกรรมคำสั่ง (Prompt Strategy) และโครงสร้างตัวอย่าง (Few-shot)
    prompt = f"""<instruction>
คุณเป็นพนักงานร้านค้าอิเล็กทรอนิกส์ "ฟ้าใหม่" (FahMai) ระดับอัจฉริยะ (ใช้ Model: Pathumma Think)
กฎเหล็กสำหรับการตอบวิเคราะห์:
1. "คิดวิเคราะห์หาจุดเชื่อมโยง" (Chain-of-Thought) ก่อนตอบเสมอ พิมพ์เหตุผลการวิเคราะห์และเทียบสเปคลงในแท็ก <think>...</think> อย่างละเอียด
2. ห้ามตอบข้อ 9 ("ไม่มีข้อมูล") หรือ 10 ("ไม่เกี่ยวข้อง") เด็ดขาด หากคำถามมีชื่อสินค้าของร้าน (เช่น สายฟ้า R1, MegaCool) แม้ลูกค้าจะเล่าเรื่องส่วนตัวยาวแค่ไหนก็ตาม! ให้ถือว่าอยู่ในหมวดร้านประยุกต์ตอบได้
3. ตอบข้อ 10 ก็ต่อเมื่อเป็นคำถามที่งี่เง่าและหลุดโลกจริงๆ เช่น "ขอสูตรทำอาหาร", "ดอกเบี้ยธนาคาร", "จองโรงแรมภูเก็ตที่ไหนดี" (โดยไม่มีการเอ่ยถึงสินค้าไอที)
4. การคำนวณค่าส่ง:
   - สินค้าหนัก >30kg จำเป็นต้องบวกเพิ่ม 200 บาท
   - ขนขึ้นชั้น 4 ขึ้นไปแบบไม่มีลิฟต์ ต้องบวกชั้นละ 100 บาท (เช่น ชั้น 6 คือบวก 3 ชั้น คือ 300 บาท)
   * ค่าส่งทั้งหมดนี้ตั้งต้นจาก 0 หากในบริบทไม่ได้ระบุค่าส่งพื้นฐาน (ฟรี)
5. เมื่อได้คำตอบ "ให้พิมพ์แค่หมายเลขข้อตัวเลขเดียวโดดๆ" ลงในแท็ก <answer>X</answer> โดยที่ X คือ 1-10
ตัวอย่างการคิด (Mega Few-Shot Pattern):
[กรณีคำนวณซับซ้อน]
คำถาม: "สั่งซื้อตู้เย็น MegaCool หนัก 40kg ไปชั้น 5 ไม่มีลิฟต์ ค่าจัดส่งเท่าไหร่"
<think>
- สินค้าหนัก 40kg (เกิน 30kg) -> บวกค่าส่งของหนัก = 200
- ขนขึ้นชั้น 5 ไม่มีลิฟต์ -> ฟรีสิทธิ์ชั้น 1-3 เหลือส่วนเกินคือชั้น 4 และชั้น 5 รวม 2 ชั้น -> 2 * 100 = 200
- รวมค่าจัดส่ง = 200 + 200 = 400
- 400 บาทตรงกับตัวเลือกข้อ 3
</think>
<answer>3</answer>
[กรณีคำถามหลอก: เล่าเรื่องยาวแต่ถามสินค้า]
กฎ: ถ้ามีชื่อสินค้าร้าน → ห้ามตอบ 9/10 เด็ดขาด!
</instruction>
<context>
{context}
</context>
<question>
{question}
</question>
<choices>
{choices_text}
</choices>
"""
    messages = [{'role': 'user', 'content': prompt}]
    # ลาก limit token เพิ่มให้คิดออก
    # 3. รอคอยการประมวลผล Chain-of-Thought จากสมอง AI
    # 3. ขั้นตอนการประมวลผลการวิเคราะห์หาคำตอบขั้นตอนความคิด (Chain-of-Thought Inference)
    response = ask_llm(messages, max_new_tokens=1024, temperature=0.0)
    # กรองเอาแค่ตัวเลขใน <answer>X</answer>
    import re
    ans_match = re.search(r'<answer>.*?(\d{1,2}).*?</answer>', response, flags=re.IGNORECASE | re.DOTALL)
    
    if ans_match:
        n_int = int(ans_match.group(1))
        if 1 <= n_int <= 10: return n_int
    # Fallback ถ้า AI ไม่ยอมปิด tag answer ให้ดึงเลขโดดๆ หลัง think หริอสุดท้าย 
    clean = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    if '<think>' in clean: clean = clean.split('<think>')[0].strip()
    if not clean: clean = response[-50:]
        
    numbers = re.findall(r'\b(\d{1,2})\b', clean)
    for n in reversed(numbers):
        n_int = int(n)
        if 1 <= n_int <= 10: return n_int
    print(f'  ⚠️ ข้อ {qid}: parse ไม่ได้ → default 9')
    return 9


SyntaxError: unterminated f-string literal (detected at line 37) (2730847476.py, line 37)

## ❓ 10. การนำเข้าและเตรียมข้อมูลทดสอบ (Test-Set Ingestion)
การจัดเตรียมชุดข้อมูลโจทย์ปัญหาเพื่อนำเข้าสู่กระบวนการประมวลผลคำตอบของระบบตามลำดับ


In [10]:
df_questions = pd.read_csv('data/questions.csv')
print(f'จำนวนคำถาม: {len(df_questions)} ข้อ')
# ทดสอบ 1 ข้อ
test_row = df_questions.iloc[0]
print(f'\nคำถาม: {test_row["question"]}')
answer = answer_mcq(test_row)
print(f'คำตอบ: ตัวเลือก {answer}')
print(f'ตัวเลือกที่เลือก: {test_row[f"choice_{answer}"]}')


จำนวนคำถาม: 100 ข้อ

คำถาม: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
คำตอบ: ตัวเลือก 10
ตัวเลือกที่เลือก: คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


## 🚀 ขั้นตอนที่ 11: การควบคุมกระบวนการประมวลผลคำตอบและประมวลผลคำตอบ (Response Orchestration)
**[Architecture Point 5: Chain-of-Thought Inference]**  
ขั้นตอนสุดท้ายในการสร้างคำตอบ โดยมีการใช้เทคนิค Few-shot Prompting และบังคับให้โมเดลประมวลผลผ่านลำดับความคิด (Chain-of-Thought) เพื่อป้องกันข้อผิดพลาดและเพิ่มความสามารถในการตัดสินใจเชิงลอจิก


In [11]:
print('🚀 เริ่มตอบทั้ง 101 ข้อ...\n')
results = []
for idx, row in df_questions.iterrows():
    qid = row['id']
    q_preview = row['question'][:60]
    print(f'🔄 [{qid:3d}/{len(df_questions)}] {q_preview}...', end=' ')
    try:
        answer = answer_mcq(row, top_k=15, use_rewrite=True)
    except Exception as e:
        print(f'❌ Error: {e}')
        answer = 9  # default fallback
    results.append({'id': qid, 'answer': answer})
    print(f'→ {answer}')
# สร้าง submission CSV
df_submission = pd.DataFrame(results)
df_submission.to_csv('submission.csv', index=False)
print(f'\n🎉 เสร็จแล้ว! ไฟล์ submission.csv สร้างเรียบร้อย ({len(df_submission)} ข้อ)')
print('\n📊 สรุปคำตอบ:')
print(df_submission['answer'].value_counts().sort_index())
print('\n👀 Preview:')
print(df_submission.head(10))


🚀 เริ่มตอบทั้ง 101 ข้อ...

🔄 [  1/100] Watch S3 Ultra กันน้ำได้กี่ ATM ครับ... → 10
🔄 [  2/100] ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ... → 7
🔄 [  3/100] หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ... → 3
🔄 [  4/100] อยากเอามือถือเครื่องเก่ามาเทิร์น เปลี่ยนเป็นเครื่องใหม่ที่ฟ้... → 6
🔄 [  5/100] สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ... → 6
🔄 [  6/100] ฟ้าใหม่จ่ายด้วย crypto ได้มั้ยคะ เช่น Bitcoin... 

KeyboardInterrupt: 

## 🏁 สรุปทางเทคนิค (Functional Implementation Summary)
เทคโนโลยี RAG ที่นำมาประยุกต์ใช้นี้แสดงถึงการผสานพลังระหว่างการจัดการบริบทเชิงลึก (Contextual Retrieval) และระบบสืบค้นข้อมูลแบบผสมผสาน (Hybrid Indexing) เพื่อให้โมเดลภาษาขนาดใหญ่ทำงานได้อย่างมีประสิทธิภาพสูงสุดภายใต้สถานการณ์ที่มีความซับซ้อนของข้อมูล
